# Probability Models, Confidence Intervals and Hypothesis Testing

**STA 532 2.0 Spreadsheet Modelling** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta532.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Probability rules and the modelling step](#theory-01)
- [Binomial model](#theory-02)
- [Poisson model](#theory-03)
- [Normal model and standard-normal tables](#theory-04)
- [Sampling distributions and t](#theory-05)
- [Confidence intervals for means](#theory-06)
- [Proportions and sample-size planning](#theory-07)
- [Hypothesis-test workflow](#theory-08)
- [Mean and proportion statistics](#theory-09)
- [Error types and interpretation](#theory-10)

## Detailed theory · Probability models, intervals and hypothesis tests

<a id="theory-01"></a>

### Probability rules and the modelling step
Define outcomes, events, the sample space and independence assumptions. Conditional probabilities use the conditioning-event denominator. A PMF gives discrete point probabilities; a PDF gives density whose integral is probability. A distribution name is justified by a data-generating mechanism, not by the appearance of a software menu.

<a id="theory-02"></a>

### Binomial model
With fixed n independent trials and common success probability p,
$$P(X=k)=\binom nkp^k(1-p)^{n-k},\quad E[X]=np,\quad Var(X)=np(1-p).$$
Exact, at-most and at-least events have different CDF endpoints. A finite-population sample without replacement is not exactly binomial unless its design justifies that approximation.

<a id="theory-03"></a>

### Poisson model
For a count in a fixed exposure with rate $\lambda$,
$$P(X=k)=e^{-\lambda}\lambda^k/k!,\quad E[X]=Var(X)=\lambda.$$
A homogeneous Poisson-process interpretation assumes independent increments and a stable event rate; overdispersion, clustering or changing exposure can invalidate it. Scale the mean to the stated exposure before calculating.

<a id="theory-04"></a>

### Normal model and standard-normal tables
For $X\sim N(\mu,\sigma^2)$, $Z=(X-\mu)/\sigma$ is standard normal. Interval probabilities subtract two CDF values. Check whether a supplied normal table reports left-tail probability, upper-tail probability or an area between zero and z. Inverse CDF calculations turn target probabilities into quantiles.

<a id="theory-05"></a>

### Sampling distributions and t
The distribution of observations differs from the distribution of their sample mean. For independent observations, $SE(\bar X)=\sigma/\sqrt n$. Normal data give an exact normal mean distribution; a central-limit approximation requires adequate conditions. When SD is estimated from a normal sample,
$$T=(\bar X-\mu)/(S/\sqrt n)\sim t_{n-1}.$$
The t distribution has heavier tails and approaches normal as df grows.

<a id="theory-06"></a>

### Confidence intervals for means
Known population SD uses $\bar x\pm z_{1-\alpha/2}\sigma/\sqrt n$. Unknown SD with the normal-sample model uses $\bar x\pm t_{1-\alpha/2,n-1}s/\sqrt n$. Confidence describes the repeated-sampling method's coverage, not the fraction of individual values inside the interval.

<a id="theory-07"></a>

### Proportions and sample-size planning
The familiar Wald interval is $\hat p\pm z\sqrt{\hat p(1-\hat p)/n}$, but it can perform poorly at small n or probabilities near a boundary. Use an interval appropriate for the counts rather than clipping an unsuitable interval without explanation. Approximate planning uses
$$n_\mu=\lceil z^2\sigma^2/E^2\rceil,\quad n_p=\lceil z^2p(1-p)/E^2\rceil.$$
Use a plausible planning SD or proportion; p=.5 is conservative for absolute proportion precision. Include design and nonresponse adjustments when relevant.

<a id="theory-08"></a>

### Hypothesis-test workflow
State a population parameter, H0 and H1, alpha, assumptions, statistic and reference distribution. Compute the statistic, then use a matching critical region or p-value. Choose tails before data inspection. A p-value is a null-model tail probability, not the probability the null is true.

<a id="theory-09"></a>

### Mean and proportion statistics
For a known-SD mean use $Z=(\bar x-\mu_0)/(\sigma/\sqrt n)$; for unknown SD use the corresponding t statistic. A null proportion test uses $\sqrt{p_0(1-p_0)/n}$, not the interval's estimated SE, in its usual normal statistic. Small counts can use an exact binomial test.

<a id="theory-10"></a>

### Error types and interpretation
Alpha controls Type I error under the model; beta depends on a specified alternative and power is 1−beta. Non-rejection is not proof of no effect. Report estimates, intervals and practical importance with the decision. A spreadsheet template records the argument; it cannot select or validate the model for you.

## Concepts covered

**Course points:** Confidence intervals for means/proportions; t distribution; margin of error; sample size; tests and p-values.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Probability model, confidence interval and test: substitution table

### Question and assumptions

A normal population has known SD 10. A sample of n=25 gives mean 54. Estimate the mean and test $H_0:\mu=50$ two-sided at 5%.

### Procedure

1. Choose the known-SD normal procedure and state its conditions.
2. Calculate SE and the 95% z interval.
3. Calculate the z statistic and two-sided tail probability. Show that the CI and test agree.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Step | Substitution | Value |
| --- | --- | --- |
| SE | 10 / sqrt(25) | 2 |
| 95% critical z | Phi inverse(.975) | 1.95996 |
| Margin | z critical × SE | 3.91993 |
| Lower bound | 54 − margin | 50.0801 |
| Upper bound | 54 + margin | 57.9199 |
| Test statistic | (54−50)/SE | 2 |
| Two-sided p | 2(1−Phi(abs(z))) | 0.0455003 |

| Decision route | Conclusion |
| --- | --- |
| p≤.05? | Reject H0 |
| 50 outside CI? | True |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


n=25; xb=54; sigma=10; mu0=50; se=sigma/sqrt(n); zcrit=NormalDist().inv_cdf(.975)
z=(xb-mu0)/se; lower=xb-zcrit*se; upper=xb+zcrit*se; p=2*(1-NormalDist().cdf(abs(z)))
show_table(["Step","Substitution","Value"],[["SE","10 / sqrt(25)",se],["95% critical z","Phi inverse(.975)",zcrit],["Margin","z critical × SE",zcrit*se],["Lower bound","54 − margin",lower],["Upper bound","54 + margin",upper],["Test statistic","(54−50)/SE",z],["Two-sided p","2(1−Phi(abs(z)))",p]])
show_table(["Decision route","Conclusion"],[["p≤.05?","Reject H0" if p<=.05 else "Do not reject H0"],["50 outside CI?",not lower<=mu0<=upper]])
assert (p<=.05)==(not lower<=mu0<=upper)


### Decision and interpretation

Known population SD is essential here. If SD is estimated from a small normal sample, use a t procedure. The interval concerns the population mean, not the range containing 95% of observations.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
pop=rng.normal(500,12,100000); mu=pop.mean(); cover=[]
for _ in range(1500):
 s=rng.choice(pop,25); se=s.std(ddof=1)/5; t=stats.t.ppf(.975,24); cover.append(s.mean()-t*se<=mu<=s.mean()+t*se)
print('95% t interval coverage',np.mean(cover).round(3))
z=stats.norm.ppf(.975); print('n for margin 3, sigma 12:',np.ceil((z*12/3)**2))
s=rng.normal(503,8,36); t=(s.mean()-500)/(s.std(ddof=1)/6); print('one-sample t',round(t,3),'two-sided p',round(2*stats.t.sf(abs(t),35),4))

### Practice set · Uncertainty court
1. A sample mean is 503, s=8, n=36. Construct a 95% t interval for μ.
2. Interpret p=.03 for a two-sided test at α=.05.
3. Find planned n for mean margin 3, σ≈12, 95% confidence.
4. Distinguish Type I error from Type II error.

<details><summary>Answer sketches</summary>
1. 503±t(.975,35)(8/6), about (500.29,505.71). 2. If H₀/model were true, a result at least this extreme would occur about 3% of the time; reject at .05, without saying H₀ has 3% probability. 3. ceil[(1.96·12/3)²]=ceil(61.47)=62. 4. Type I is false alarm; Type II is missed effect.
</details>

### Debrief
Explain a confidence procedure and a p-value without saying the null is probably true/false.

Record the formula, assumptions, and one follow-up question.

## Lab reflection

Which assumptions came from the study design? What changed when you adjusted a parameter?